# 1. Setup and document selection


In [1]:
import importlib
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
DOCS_DIR = PROJECT_ROOT / "docs"
THEMES_PDFS_DIR = DOCS_DIR / "generated_pdfs"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

EXTRA_PDF_FILES = [
   
    DOCS_DIR / "Organiser_un_evenement_deAaZ.pdf",
    DOCS_DIR / "Checklists.pdf",
    DOCS_DIR / "Guideline_Sustainable_Event.pdf",
    DOCS_DIR / "تقاليد الزفاف في الثقافات العربية.pdf",
]

themes_pdfs = sorted(THEMES_PDFS_DIR.glob("*.pdf"))
PDF_FILES = list(dict.fromkeys(EXTRA_PDF_FILES + themes_pdfs))

missing_files = [path for path in PDF_FILES if not path.is_file()]
if missing_files:
    raise FileNotFoundError(f"Missing PDFs: {missing_files}")

print("PDFs selected:", len(PDF_FILES))
print("Individual PDFs:", len(EXTRA_PDF_FILES))
print("Theme PDFs:", len(themes_pdfs))
for path in PDF_FILES:
    print(" -", path.relative_to(PROJECT_ROOT))


PDFs selected: 20
Individual PDFs: 4
Theme PDFs: 16
 - docs\Organiser_un_evenement_deAaZ.pdf
 - docs\Checklists.pdf
 - docs\Guideline_Sustainable_Event.pdf
 - docs\تقاليد الزفاف في الثقافات العربية.pdf
 - docs\generated_pdfs\Boho_Wedding.pdf
 - docs\generated_pdfs\Celestial_Wedding.pdf
 - docs\generated_pdfs\Classic_Wedding.pdf
 - docs\generated_pdfs\Enchanted_Forest_Wedding.pdf
 - docs\generated_pdfs\Garden_Wedding.pdf
 - docs\generated_pdfs\Modern_Wedding.pdf
 - docs\generated_pdfs\Moody_Wedding.pdf
 - docs\generated_pdfs\Romantic_Wedding.pdf
 - docs\generated_pdfs\Rustic_Wedding.pdf
 - docs\generated_pdfs\Tropical_Wedding.pdf
 - docs\generated_pdfs\Vintage_Wedding.pdf
 - docs\generated_pdfs\Wedding_Centerpieces.pdf
 - docs\generated_pdfs\Wedding_Colors.pdf
 - docs\generated_pdfs\Wedding_Themes.pdf
 - docs\generated_pdfs\Wedding_Trends.pdf
 - docs\generated_pdfs\Whimsical_Wedding.pdf


# 2. Chunking strategies


In [2]:
STRATEGY_CONFIGS = {
    "fixed_size": {
        "enabled": True,
        "kwargs": {"chunk_size": 800, "overlap": 100},
    },
    "recursive": {
        "enabled": True,
        "kwargs": {"chunk_size": 800, "overlap": 100},
    },
    "sentence": {
        "enabled": True,
        "kwargs": {"sentences_per_chunk": 4, "sentence_overlap": 1},
    },
    "structure_aware": {
        "enabled": True,
        "kwargs": {"chunk_size": 800, "overlap": 100},
    },
    "semantic": {
        "enabled": True,
        "kwargs": {
            "similarity_threshold": 0.65,
            "max_chunk_size": 800,
            "unit_prefix": "passage: ",
        },
    },
}

enabled_strategies = {
    name: config["kwargs"]
    for name, config in STRATEGY_CONFIGS.items()
    if config["enabled"]
}

print("Methods that will run:")
for name, settings in enabled_strategies.items():
    print(f" - {name}: {settings}")


Methods that will run:
 - fixed_size: {'chunk_size': 800, 'overlap': 100}
 - recursive: {'chunk_size': 800, 'overlap': 100}
 - sentence: {'sentences_per_chunk': 4, 'sentence_overlap': 1}
 - structure_aware: {'chunk_size': 800, 'overlap': 100}
 - semantic: {'similarity_threshold': 0.65, 'max_chunk_size': 800, 'unit_prefix': 'passage: '}


# 3. Import `doc_rag` helpers and load the embedding model


In [3]:
extraction_module = importlib.import_module("doc_rag.2_extraction")
chunking_module = importlib.import_module("doc_rag.4_chunking")
embeddings_module = importlib.import_module("doc_rag.5_embeddings")

extraction_module = importlib.reload(extraction_module)
chunking_module = importlib.reload(chunking_module)
embeddings_module = importlib.reload(embeddings_module)

extract_pdf = extraction_module.extract_pdf
chunk_extracted_documents = chunking_module.chunk_extracted_documents
load_embedding_model = embeddings_module.load_embedding_model
prepare_texts = embeddings_module.prepare_texts
embed_texts = embeddings_module.embed_texts
embed_queries = embeddings_module.embed_queries
cosine_search = embeddings_module.cosine_search

EMBEDDING_BATCH_SIZE = embeddings_module.DEFAULT_BATCH_SIZE


RETRIEVAL_MODEL_NAME = RETRIEVAL_MODEL_NAME = "intfloat/multilingual-e5-base"

CHUNKING_MODEL_NAME = "intfloat/multilingual-e5-base"

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for semantic chunking and retrieval eval.")

retrieval_model = load_embedding_model(
    RETRIEVAL_MODEL_NAME,
    device="cuda",
    local_files_only=True,
    model_kwargs={"torch_dtype": torch.float16},
)
retrieval_model.max_seq_length = embeddings_module.DEFAULT_MAX_SEQUENCE_LENGTH

chunking_model = load_embedding_model(
    CHUNKING_MODEL_NAME,
    device="cuda",
    local_files_only=True,
    model_kwargs={"torch_dtype": torch.float16},
)
chunking_model.max_seq_length = embeddings_module.DEFAULT_MAX_SEQUENCE_LENGTH

# keep old names so later retrieval cells still work
embedding_model = retrieval_model
EMBEDDING_MODEL_NAME = RETRIEVAL_MODEL_NAME

print("Chunking model (semantic splits):", CHUNKING_MODEL_NAME)
print("Retrieval model (eval embeddings):", EMBEDDING_MODEL_NAME)
print("Device:", next(embedding_model.parameters()).device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Chunking model (semantic splits): intfloat/multilingual-e5-base
Retrieval model (eval embeddings): intfloat/multilingual-e5-base
Device: cuda:0


# 4. Extract text from every selected PDF


In [4]:
extracted_documents = []

for position, pdf_file in enumerate(PDF_FILES, start=1):
    print(f"[{position}/{len(PDF_FILES)}] Extracting {pdf_file.name}")
    extracted_documents.append(extract_pdf(pdf_file))

extraction_summary = pd.DataFrame([
    {
        "file_name": document["file_name"],
        "pages": document["page_count"],
        "pages_with_text": sum(
            bool(page["final_text"].strip()) for page in document["pages"]
        ),
        "final_text_characters": sum(
            len(page["final_text"]) for page in document["pages"]
        ),
    }
    for document in extracted_documents
])
display(extraction_summary)


[1/20] Extracting Organiser_un_evenement_deAaZ.pdf
[2/20] Extracting Checklists.pdf
[3/20] Extracting Guideline_Sustainable_Event.pdf
[4/20] Extracting تقاليد الزفاف في الثقافات العربية.pdf
[5/20] Extracting Boho_Wedding.pdf
[6/20] Extracting Celestial_Wedding.pdf
[7/20] Extracting Classic_Wedding.pdf
[8/20] Extracting Enchanted_Forest_Wedding.pdf
[9/20] Extracting Garden_Wedding.pdf
[10/20] Extracting Modern_Wedding.pdf
[11/20] Extracting Moody_Wedding.pdf
[12/20] Extracting Romantic_Wedding.pdf
[13/20] Extracting Rustic_Wedding.pdf
[14/20] Extracting Tropical_Wedding.pdf
[15/20] Extracting Vintage_Wedding.pdf
[16/20] Extracting Wedding_Centerpieces.pdf
[17/20] Extracting Wedding_Colors.pdf
[18/20] Extracting Wedding_Themes.pdf
[19/20] Extracting Wedding_Trends.pdf
[20/20] Extracting Whimsical_Wedding.pdf


,file_name,pages,pages_with_text,final_text_characters
0,Organiser_un_evenement_deAaZ.pdf,34,34,50999
1,Checklists.pdf,10,10,8676
2,Guideline_Sustainable_Event.pdf,22,22,44330
3,تقاليد الزفاف في الثقافات العربية.pdf,39,38,53744
4,Boho_Wedding.pdf,32,32,32658
5,Celestial_Wedding.pdf,13,13,6412
6,Classic_Wedding.pdf,16,16,8959
7,Enchanted_Forest_Wedding.pdf,23,23,24185
8,Garden_Wedding.pdf,25,25,15089
9,Modern_Wedding.pdf,30,30,39708


## 5. Generate chunks with every enabled method


In [5]:
chunks_by_strategy = {}

for strategy_name, strategy_kwargs in enabled_strategies.items():
    print(f"Chunking with {strategy_name}: {strategy_kwargs}")
    runtime_kwargs = dict(strategy_kwargs)
    if strategy_name == "semantic":
        runtime_kwargs["model"] = chunking_model

    chunks_by_strategy[strategy_name] = chunk_extracted_documents(
        extracted_documents,
        strategy=strategy_name,
        **runtime_kwargs,
    )
    print("  chunks created:", len(chunks_by_strategy[strategy_name]))


Chunking with fixed_size: {'chunk_size': 800, 'overlap': 100}
  chunks created: 786
Chunking with recursive: {'chunk_size': 800, 'overlap': 100}
  chunks created: 787
Chunking with sentence: {'sentences_per_chunk': 4, 'sentence_overlap': 1}
  chunks created: 3581
Chunking with structure_aware: {'chunk_size': 800, 'overlap': 100}
  chunks created: 880
Chunking with semantic: {'similarity_threshold': 0.65, 'max_chunk_size': 800, 'unit_prefix': 'passage: '}
  chunks created: 775


## 6. Compare chunk-size distributions


In [6]:
comparison_rows = []

for strategy_name, chunks in chunks_by_strategy.items():
    lengths = chunks["character_count"]
    comparison_rows.append({
        "strategy": strategy_name,
        "chunks": len(chunks),
        "documents": chunks["file_name"].nunique(),
        "pages": chunks[["file_name", "page_number"]].drop_duplicates().shape[0],
        "min_chars": int(lengths.min()),
        "median_chars": int(lengths.median()),
        "mean_chars": round(float(lengths.mean()), 1),
        "p90_chars": int(lengths.quantile(0.90)),
        "max_chars": int(lengths.max()),
        "short_chunks_lt_100": int((lengths < 100).sum()),
        "short_chunk_percent": round(float((lengths < 100).mean() * 100), 1),
    })

strategy_comparison = pd.DataFrame(comparison_rows).set_index("strategy")
display(strategy_comparison)


,chunks,documents,pages,min_chars,median_chars,mean_chars,p90_chars,max_chars,short_chunks_lt_100,short_chunk_percent
strategy,,,,,,,,,,
fixed_size,786,20,428,3,723,604.1,800,800,4,0.5
recursive,787,20,428,3,707,589.3,790,799,9,1.1
sentence,3581,20,428,3,149,157.6,257,470,730,20.4
structure_aware,880,20,428,3,558,509.5,776,800,51,5.8
semantic,775,20,428,2,696,564.9,792,800,27,3.5


## 7. Inspect comparable chunk samples


In [7]:
SAMPLE_FILE_NAME = "Wedding_Themes.pdf"
SAMPLE_PAGE_NUMBER = 4
SAMPLE_LIMIT_PER_STRATEGY = 3

for strategy_name, chunks in chunks_by_strategy.items():
    sample = chunks[
        (chunks["file_name"] == SAMPLE_FILE_NAME)
        & (chunks["page_number"] == SAMPLE_PAGE_NUMBER)
    ].head(SAMPLE_LIMIT_PER_STRATEGY)

    
    print("METHOD:", strategy_name)
    print("FILE:", SAMPLE_FILE_NAME, "- PAGE:", SAMPLE_PAGE_NUMBER)

    if sample.empty:
        print("No chunks found for this file and page.")
        continue

    for row in sample.itertuples(index=False):
        print(f"\n[{row.chunk_id}] {row.character_count} characters")
        print(row.text)
    print("\n\n\n")


METHOD: fixed_size
FILE: Wedding_Themes.pdf - PAGE: 4

[Wedding_Themes-9d09076f38a8-p0004-c000] 800 characters
While every wedding style can certainly accommodate a variety of guest experiences, certain
themes more easily cater to particular dining, entertainment, and immersive décor arrangements.
For example, a Champagne tower will always look classic — but twist tradition with a burgundy
wine tower for your moody affair or rosé for your whimsical garden celebration.
PRO
Save countless scrolls searching through Pinterest, and find our favorite wedding themes below.
Make sure to click “See More Photos” for a deeper look at each style.
Wedding Theme & Style Ideas
1. Classic Weddings That Stand the Test of Time
Opt for a timeless aesthetic with a classic
wedding style. Lean into simple color palettes,
round or posy-shaped floral arrangements, and
elegant furnishings that will always be popular
— rega

[Wedding_Themes-9d09076f38a8-p0004-c001] 758 characters
round or posy-shaped floral arr

## 8. Integrity checks


In [8]:
integrity_rows = []

for strategy_name, chunks in chunks_by_strategy.items():
    config = enabled_strategies[strategy_name]
    configured_limit = config.get("chunk_size", config.get("max_chunk_size"))
    limit_respected = (
        True
        if configured_limit is None
        else bool(chunks["character_count"].le(configured_limit).all())
    )
    integrity_rows.append({
        "strategy": strategy_name,
        "nonempty_text": bool(chunks["text"].str.strip().ne("").all()),
        "unique_chunk_ids": bool(chunks["chunk_id"].is_unique),
        "valid_page_numbers": bool(chunks["page_number"].ge(1).all()),
        "source_files_exist": bool(
            chunks["source_path"].map(lambda value: Path(value).is_file()).all()
        ),
        "configured_size_limit_respected": limit_respected,
    })

integrity_checks = pd.DataFrame(integrity_rows).set_index("strategy")
display(integrity_checks)
assert integrity_checks.to_numpy().all(), "At least one chunk integrity check failed."
print("All integrity checks passed.")


,nonempty_text,unique_chunk_ids,valid_page_numbers,source_files_exist,configured_size_limit_respected
strategy,,,,,
fixed_size,True,True,True,True,True
recursive,True,True,True,True,True
sentence,True,True,True,True,True
structure_aware,True,True,True,True,True
semantic,True,True,True,True,True


All integrity checks passed.


## 9. Retrieval evaluation

Use `document_chunking_cases.json`

Embed queries and chunks with `doc_rag.5_embeddings`

rank by cosine similarity 

A hit requires matching `expected_file` and one of `expected_pages`

In [9]:
EVALUATION_CASES_FILE = (
    PROJECT_ROOT
    / "data"
    / "rag"
    / "experiments"
    / "chunking_evaluation"
    / "document_chunking_cases.json"
)
EVALUATION_RESULTS_FILE = (
    PROJECT_ROOT
    / "data"
    / "rag"
    / "experiments"
    / "chunking_evaluation"
    / "document_chunking_results.json"
)

with EVALUATION_CASES_FILE.open(encoding="utf-8") as file:
    evaluation_payload = json.load(file)

evaluation_cases = evaluation_payload["cases"]
query_embeddings = embed_queries(
    [case["query"] for case in evaluation_cases],
    model=embedding_model,
    model_name=EMBEDDING_MODEL_NAME,
    batch_size=EMBEDDING_BATCH_SIZE,
)
print("Evaluation cases loaded:", len(evaluation_cases))


Evaluation cases loaded: 42


In [10]:
retrieval_rows = []
strategy_metric_rows = []

for strategy_name, chunks in chunks_by_strategy.items():
    print(f"Embedding chunks for retrieval evaluation: {strategy_name}")
    prepared_chunks = prepare_texts(
        chunks["text"].tolist(),
        model_name=EMBEDDING_MODEL_NAME,
        text_type="passage",
    )
    chunk_embeddings = embed_texts(
        prepared_chunks,
        embedding_model,
        batch_size=EMBEDDING_BATCH_SIZE,
    )
    scores, positions = cosine_search(
        query_embeddings,
        chunk_embeddings,
        top_k=5,
    )

    reciprocal_ranks = []
    hits_at_1 = []
    hits_at_3 = []
    hits_at_5 = []

    for query_index, case in enumerate(evaluation_cases):
        expected_pages = set(case["expected_pages"])
        first_relevant_rank = None
        top_results = []

        for rank, position in enumerate(positions[query_index], start=1):
            row = chunks.iloc[int(position)]
            relevant = (
                row["file_name"] == case["expected_file"]
                and int(row["page_number"]) in expected_pages
            )
            if relevant and first_relevant_rank is None:
                first_relevant_rank = rank
            top_results.append({
                "rank": rank,
                "chunk_id": row["chunk_id"],
                "file_name": row["file_name"],
                "page_number": int(row["page_number"]),
                "score": round(float(scores[query_index, rank - 1]), 6),
                "relevant": bool(relevant),
            })

        rank = first_relevant_rank
        reciprocal_ranks.append(0.0 if rank is None else 1.0 / rank)
        hits_at_1.append(rank is not None and rank <= 1)
        hits_at_3.append(rank is not None and rank <= 3)
        hits_at_5.append(rank is not None and rank <= 5)
        retrieval_rows.append({
            "strategy": strategy_name,
            "case_id": case["id"],
            "query": case["query"],
            "expected_file": case["expected_file"],
            "expected_pages": case["expected_pages"],
            "first_relevant_rank": rank,
            "top_5": top_results,
        })

    strategy_metric_rows.append({
        "strategy": strategy_name,
        "recall_at_1": round(float(np.mean(hits_at_1)), 4),
        "recall_at_3": round(float(np.mean(hits_at_3)), 4),
        "recall_at_5": round(float(np.mean(hits_at_5)), 4),
        "mrr_at_5": round(float(np.mean(reciprocal_ranks)), 4),
    })

retrieval_details = pd.DataFrame(retrieval_rows)
retrieval_metrics = pd.DataFrame(strategy_metric_rows).set_index("strategy")
display(
    retrieval_metrics.sort_values(
        ["recall_at_5", "mrr_at_5"],
        ascending=False,
    )
)


Embedding chunks for retrieval evaluation: fixed_size


Batches:   0%|          | 0/786 [00:00<?, ?it/s]

Embedding chunks for retrieval evaluation: recursive


Batches:   0%|          | 0/787 [00:00<?, ?it/s]

Embedding chunks for retrieval evaluation: sentence


...

Embedding chunks for retrieval evaluation: structure_aware


Batches:   0%|          | 0/880 [00:00<?, ?it/s]

Embedding chunks for retrieval evaluation: semantic


Batches:   0%|          | 0/775 [00:00<?, ?it/s]

,recall_at_1,recall_at_3,recall_at_5,mrr_at_5
strategy,,,,
semantic,0.5238,0.7857,0.8333,0.6548
fixed_size,0.5714,0.7143,0.7857,0.6492
structure_aware,0.5238,0.7143,0.7857,0.6266
recursive,0.5952,0.7619,0.7619,0.6706
sentence,0.4762,0.7143,0.7381,0.5893


In [11]:
failed_cases = retrieval_details[
    retrieval_details["first_relevant_rank"].isna()
]
display(
    failed_cases[
        ["strategy", "case_id", "query", "expected_file", "expected_pages"]
    ]
)

results_payload = {
    "schema_version": 1,
    "embedding_model": EMBEDDING_MODEL_NAME,
    "evaluation_case_file": str(
        EVALUATION_CASES_FILE.relative_to(PROJECT_ROOT)
    ),
    "strategy_configs": enabled_strategies,
    "retrieval_metrics": retrieval_metrics.reset_index().to_dict(
        orient="records"
    ),
    "chunk_statistics": strategy_comparison.reset_index().to_dict(
        orient="records"
    ),
    "case_results": retrieval_rows,
}
EVALUATION_RESULTS_FILE.parent.mkdir(parents=True, exist_ok=True)
with EVALUATION_RESULTS_FILE.open("w", encoding="utf-8") as file:
    json.dump(results_payload, file, ensure_ascii=False, indent=2)
print("Saved evaluation results to:", EVALUATION_RESULTS_FILE)


,strategy,case_id,query,expected_file,expected_pages
7,fixed_size,modern_acrylic,How can acrylic menus and black-and-white colo...,Modern_Wedding.pdf,[15]
21,fixed_size,organiser_role,What does an event organizer actually do when ...,Organiser_un_evenement_deAaZ.pdf,"[5, 6]"
22,fixed_size,organiser_logistics,"Who is responsible for managing suppliers, equ...",Organiser_un_evenement_deAaZ.pdf,[11]
23,fixed_size,organiser_regisseur,"Qui s’occupe de vérifier que le son, l’éclaira...",Organiser_un_evenement_deAaZ.pdf,[12]
24,fixed_size,organiser_translation,Si certains participants ne comprennent pas la...,Organiser_un_evenement_deAaZ.pdf,[17]
25,fixed_size,organiser_agency,Quel est le rôle d’une agence événementielle e...,Organiser_un_evenement_deAaZ.pdf,[18]
27,fixed_size,checklists_venue_transport,I'm choosing a venue for my event. What should...,Checklists.pdf,"[3, 4]"
28,fixed_size,checklists_energy_water,What energy and water savings measures should ...,Checklists.pdf,[5]
31,fixed_size,guideline_minimum_criteria,I want to organize a sustainable event but I c...,Guideline_Sustainable_Event.pdf,[3]
49,recursive,modern_acrylic,How can acrylic menus and black-and-white colo...,Modern_Wedding.pdf,[15]


Saved evaluation results to: c:\Users\User\Desktop\inmind\gatherly_rag\data\rag\experiments\chunking_evaluation\document_chunking_results.json
